# EGU-Net-ss on Samson
Run cells from top to bottom. Ground-truth variables are used only in evaluation.


In [ ]:
from pathlib import Path
import sys, json, types
# Works whether Jupyter starts in the project root or in code/.
HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == 'code' else HERE
if not (ROOT / 'code' / 'settings.py').exists():
    raise RuntimeError('Open this notebook from EGU-Net-ss-Simple/code or its project root.')
# Use a unique package name; the required folder name `code` conflicts with Python's stdlib module.
egu_simple = types.ModuleType('egu_simple')
egu_simple.__path__ = [str(ROOT / 'code')]
sys.modules['egu_simple'] = egu_simple
from egu_simple import settings as S
from egu_simple.data_loader import load_samson
from egu_simple.preprocessing import preprocess
from egu_simple.egu_net_ss import EGUNetSS
from egu_simple.trainer import select_device, train
from egu_simple.checkpoint import load_checkpoint
from egu_simple.evaluation import evaluate_and_save
from egu_simple.visualization import save_evaluation_figures
import numpy as np, torch, matplotlib.pyplot as plt


## 1. Environment and device


In [ ]:
print('PyTorch:', torch.__version__)
device, diagnostic = select_device(S.DEVICE)
print('Selected device:', device)
print(diagnostic)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(device))
    free, total = torch.cuda.mem_get_info(device)
    print(f'Available GPU memory: {free/2**30:.2f} / {total/2**30:.2f} GiB')
else:
    raise RuntimeError('Full Samson training was not started because CUDA is unusable. Use the small CPU smoke test; for full training, install a PyTorch build compatible with the NVIDIA driver, then restart the kernel.')


## 2. Load and inspect Samson


In [ ]:
data = load_samson(S.HYPERSPECTRAL_PATH, S.GROUND_TRUTH_PATH,
                   (S.IMAGE_HEIGHT, S.IMAGE_WIDTH, S.SPECTRAL_BANDS), S.NUM_MATERIALS)
cube = data['cube']; spectra = data['spectra']
print('Cube H×W×B:', cube.shape, '| V bands×pixels:', spectra.shape)
print('Evaluation-only GT A:', data['abundances_gt'].shape, '| M:', data['endmembers_gt'].shape)


## 3. Cached, GT-independent pseudo-endmember preprocessing


In [ ]:
prep = preprocess(spectra, data['shape'], S.PROCESSED_DIR, S.NUM_MATERIALS,
                  S.BLOCK_SIZE, S.LOCAL_VCA_VERTICES, S.BUNDLE_CLUSTERS, S.RANDOM_SEED)
print('Pseudo-endmembers:', prep['bundles'].shape, '| pseudo-labels:', prep['pseudo_labels'].shape)


## 4–5. Initialize and train/resume
The normal epoch target is in `code/settings.py`. Re-running this cell resumes `latest.pt`; a completed run is left complete. Reduce `BATCH_SIZE` if GPU memory is limited. If CUDA fails, install a PyTorch build compatible with the installed NVIDIA driver once, outside this notebook.


In [ ]:
model = EGUNetSS(S.SPECTRAL_BANDS, S.NUM_MATERIALS, S.DROPOUT_RATE)
print(f'Trainable parameters: {model.parameter_count():,}; sharing valid: {model.sharing_is_valid()}')
config = {'bands': S.SPECTRAL_BANDS, 'materials': S.NUM_MATERIALS, 'dropout_rate': S.DROPOUT_RATE,
          'learning_rate': S.LEARNING_RATE, 'poly_power': S.POLY_POWER,
          'l2_coefficient': S.L2_COEFFICIENT, 'reconstruction_weight': S.RECONSTRUCTION_WEIGHT,
          'mixture_reconstruction_weight': S.MIXTURE_RECONSTRUCTION_WEIGHT,
          'abundance_supervision_weight': S.ABUNDANCE_SUPERVISION_WEIGHT,
          'mixed_precision': S.MIXED_PRECISION, 'batch_size': S.BATCH_SIZE}
pixel_abundance_targets = np.stack([
    prep['pixel_abundances'][:, i].reshape(S.IMAGE_HEIGHT, S.IMAGE_WIDTH, order='F')
    for i in range(S.NUM_MATERIALS)
], axis=0)[None, ...].astype(np.float32)
history = train(model, prep['bundles'], prep['pseudo_labels'], cube, S.CHECKPOINT_DIR, config,
                device, S.EPOCHS, S.BATCH_SIZE, S.CHECKPOINT_EVERY_STEPS, resume=True,
                endmembers=prep['prototypes'],
                abundance_targets=pixel_abundance_targets)
print('Saved recovery checkpoint:', S.CHECKPOINT_DIR / 'latest.pt')


## 6. Evaluation and feature export
Channel matching uses reference abundances only after training has finished. The nonnegative endmember fit is also evaluation-only.


In [ ]:
model.eval()
with torch.no_grad():
    x = torch.as_tensor(prep['bundles'], dtype=torch.float32, device=device)
    scene = torch.as_tensor(cube, dtype=torch.float32, device=device).permute(2,0,1).unsqueeze(0)
    result = model(x, scene, return_features=True)
    predicted = result['abundances'][0].cpu().numpy()
    reconstructed = result['reconstruction'][0].permute(1,2,0).cpu().numpy()
    features = {k: v.detach().cpu().numpy() for k, v in result['features'].items()}
S.FEATURE_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(S.FEATURE_DIR / 'selected_features.npz', **features)
summary, abundance_maps, recovered, reconstructed = evaluate_and_save(
    spectra, data['shape'], predicted, reconstructed, data['abundances_gt'],
    data['endmembers_gt'], S.OUTPUT_DIR, S.MATERIAL_NAMES)


## 7–8. Required graphs and metrics


In [ ]:
figure_paths = save_evaluation_figures(data['abundances_gt'], abundance_maps,
    data['endmembers_gt'], recovered, history, data['shape'], S.MATERIAL_NAMES, S.FIGURE_DIR)
for path in figure_paths:
    display(plt.imread(path))
print(json.dumps(summary, indent=2))


## 9. Saved artifacts


In [ ]:
print('Latest checkpoint:', S.CHECKPOINT_DIR / 'latest.pt')
print('Best training-loss checkpoint:', S.CHECKPOINT_DIR / 'best.pt')
print('Figures:', S.FIGURE_DIR)
print('Metrics:', S.METRIC_DIR)
print('Predictions and recovered spectra:', S.FEATURE_DIR / 'predictions.npz')
